# Investment Agent — Diagnostics & Data Source Testing

This notebook lets you test **every API** and **every agent** independently.  
Run cells top-to-bottom, or jump to any section you want to debug.

**Sections:**
1. Setup & Config Validation
2. yfinance — OHLCV, Fundamentals, Macro Data
3. Technical Indicators
4. Finnhub — Sentiment, Analysts, Price Targets, Insiders
5. Alpha Vantage — News Sentiment
6. Trading 212 — Account, Portfolio, Instruments
7. Sub-Strategies — Momentum, Mean Reversion, Factor
8. Strategy Engine — Claude Synthesis
9. Moderation Panel — GPT-4o & Gemini Reviews
10. Risk Manager — Hard Rule Checks

---

## 1. Setup & Config Validation

In [1]:
import sys, os, json
from pathlib import Path
from pprint import pprint

# Ensure project root is on the path
PROJECT_ROOT = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# Load .env
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

Project root: /Users/Kayvan/Library/Mobile Documents/com~apple~CloudDocs/AI_Projects/Investment-agent
Python: 3.11.13 (main, Jun  5 2025, 08:14:07) [Clang 14.0.6 ]


In [2]:
# Validate that all required env vars are set (masked)
REQUIRED_KEYS = [
    "T212_API_KEY", "T212_API_SECRET",
    "ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_AI_API_KEY",
    "FINNHUB_API_KEY", "ALPHA_VANTAGE_API_KEY",
]

print("Environment variable check:")
print("-" * 40)
all_ok = True
for key in REQUIRED_KEYS:
    val = os.getenv(key)
    if val:
        masked = val[:4] + "..." + val[-4:] if len(val) > 8 else "****"
        print(f"  {key}: {masked}")
    else:
        print(f"  {key}: MISSING")
        all_ok = False

print()
print("All keys present" if all_ok else "Some keys are MISSING — affected cells will fail")

Environment variable check:
----------------------------------------
  T212_API_KEY: 3264...kLhh
  T212_API_SECRET: 74ql...XktY
  ANTHROPIC_API_KEY: sk-a...kwAA
  OPENAI_API_KEY: sk-p...vesA
  GOOGLE_AI_API_KEY: AIza...6uU8
  FINNHUB_API_KEY: d6e7...3u30
  ALPHA_VANTAGE_API_KEY: GUF9...JVZJ

All keys present


In [3]:
# Load settings.yaml
from src.utils.config import get_settings

settings = get_settings()
print("Settings loaded successfully")
print(f"  T212 base URL:  {settings.t212_base_url}")
print(f"  Strategy model: {settings.strategy_model}")
print(f"  Moderator 1:    {settings.moderator_1_model}")
print(f"  Moderator 2:    {settings.moderator_2_model}")
print(f"  Max positions:  {settings.max_positions}")
print(f"  Daily budgets:  Anthropic £{settings.anthropic_daily_gbp} | OpenAI £{settings.openai_daily_gbp} | Google £{settings.google_daily_gbp}")
print(f"  Monthly cap:    £{settings.total_monthly_gbp}")

Settings loaded successfully
  T212 base URL:  https://demo.trading212.com/api/v0
  Strategy model: claude-sonnet-4-5-20250929
  Moderator 1:    gpt-4o
  Moderator 2:    gemini-2.0-flash
  Max positions:  15
  Daily budgets:  Anthropic £1.0 | OpenAI £0.75 | Google £0.5
  Monthly cap:    £50.0


---
## 2. yfinance — OHLCV, Fundamentals, Macro Data

Free, no API key needed. This is the backbone data source.

In [4]:
# Test stock for all cells
TEST_TICKER = "AAPL"

In [5]:
# 2a. OHLCV price data
import yfinance as yf
import pandas as pd

df = yf.download(TEST_TICKER, period="1y", interval="1d", progress=False)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f"{TEST_TICKER} — {len(df)} trading days loaded")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"Latest close: ${df['Close'].iloc[-1]:.2f}")
print(f"52-week high: ${df['Close'].max():.2f}")
print(f"52-week low:  ${df['Close'].min():.2f}")
print()
df.tail(5)

AAPL — 251 trading days loaded
Date range: 2025-02-25 to 2026-02-24
Latest close: $272.14
52-week high: $285.92
52-week low:  $171.67



Price,Close,High,Low,Open,Volume
Date,,,,,
2026-02-18,264.350006,266.820007,262.450012,263.600006,34203300
2026-02-19,260.579987,264.480011,260.049988,262.600006,30845300
2026-02-20,264.579987,264.750000,258.160004,258.970001,42070500
2026-02-23,266.179993,269.429993,263.380005,263.489990,37308200
2026-02-24,272.140015,274.890015,267.739990,268.000000,46710423


In [47]:
# 2b. Fundamentals via yfinance
from src.agents.market_data.fundamentals import get_fundamentals

fundamentals = get_fundamentals(TEST_TICKER)
print(f"Fundamentals for {TEST_TICKER}:")
print("-" * 40)
for k, v in fundamentals.items():
    if isinstance(v, float) and v is not None:
        print(f"  {k:25s}: {v:>12.4f}")
    else:
        print(f"  {k:25s}: {v}")

Fundamentals for AAPL:
----------------------------------------
  trailing_pe              :      34.4046
  forward_pe               :      29.2650
  pb_ratio                 :      45.3718
  roe                      :       1.5202
  profit_margin            :       0.2704
  debt_equity              :     102.6300
  revenue_growth_yoy       :       0.1570
  earnings_growth          :       0.1830
  earnings_momentum_qoq    : None
  sector                   : Technology
  industry                 : Consumer Electronics
  market_cap               : 3999893815296.0000


/Users/Kayvan/Library/Caches/pypoetry/virtualenvs/investment-agent-ZTjoZHpB-py3.11/lib/python3.11/site-packages/yfinance/scrapers/fundamentals.py:33: DeprecationWarning: 'Ticker.earnings' is deprecated as not available via API. Look for "Net Income" in Ticker.income_stmt.
  warnings.warn("'Ticker.earnings' is deprecated as not available via API. Look for \"Net Income\" in Ticker.income_stmt.", DeprecationWarning)


In [8]:
# 2c. Macro data (VIX, yield spread, S&P 500 vs 200-day MA)
from src.agents.market_data.data_fetcher import DataFetcher

fetcher = DataFetcher()
macro = fetcher.get_macro_data()

print("Macro Environment:")
print("-" * 40)
print(f"  Market regime:    {macro.get('market_regime')}")
print(f"  VIX:              {macro.get('vix', 'N/A')}")
print(f"  10Y yield:        {macro.get('ten_year_yield', 'N/A')}")
print(f"  Yield spread:     {macro.get('yield_spread_10y_short', 'N/A')}")
print(f"  S&P 500:          {macro.get('sp500_price', 'N/A')}")
print(f"  S&P 200-day MA:   {macro.get('sp500_200ma', 'N/A')}")
print(f"  S&P above 200MA:  {macro.get('sp500_above_200ma', 'N/A')}")
print(f"  % above 200MA:    {macro.get('sp500_pct_above_200ma', 'N/A')}")

Macro Environment:
----------------------------------------
  Market regime:    BULL
  VIX:              19.549999237060547
  10Y yield:        4.0329999923706055
  Yield spread:     0.44499993324279785
  S&P 500:          6890.06982421875
  S&P 200-day MA:   6542.097133789062
  S&P above 200MA:  True
  % above 200MA:    5.3189777423581


---
## 3. Technical Indicators

Calculated locally from yfinance OHLCV data using the `ta` library. No API key needed.

In [9]:
from src.agents.market_data.indicators import calculate_indicators, calculate_relative_strength

indicators = calculate_indicators(df)

print(f"Technical Indicators for {TEST_TICKER}:")
print("=" * 50)

# Price
print(f"\n  Price")
print(f"  {'Current price':25s}: ${indicators.get('current_price', 0):.2f}")

# RSI
rsi = indicators.get('rsi_14', 0)
rsi_label = "OVERBOUGHT" if rsi > 70 else "OVERSOLD" if rsi < 30 else "NEUTRAL"
print(f"\n  RSI")
print(f"  {'RSI(14)':25s}: {rsi:.1f}  [{rsi_label}]")

# MACD
print(f"\n  MACD")
print(f"  {'MACD line':25s}: {indicators.get('macd_line', 0):.4f}")
print(f"  {'MACD signal':25s}: {indicators.get('macd_signal', 0):.4f}")
print(f"  {'MACD histogram':25s}: {indicators.get('macd_histogram', 0):.4f}")
print(f"  {'Bullish crossover':25s}: {indicators.get('macd_bullish_crossover')}")
print(f"  {'Bearish crossover':25s}: {indicators.get('macd_bearish_crossover')}")

# Bollinger Bands
print(f"\n  Bollinger Bands")
print(f"  {'Upper':25s}: ${indicators.get('bb_upper', 0):.2f}")
print(f"  {'Middle (20-day MA)':25s}: ${indicators.get('bb_middle', 0):.2f}")
print(f"  {'Lower':25s}: ${indicators.get('bb_lower', 0):.2f}")
print(f"  {'%B (position in band)':25s}: {indicators.get('bb_pct', 0):.2f}")
print(f"  {'Below lower band':25s}: {indicators.get('below_lower_bb')}")

# Moving Averages
print(f"\n  Moving Averages")
print(f"  {'20-day MA':25s}: ${indicators.get('ma_20', 0):.2f}")
print(f"  {'50-day MA':25s}: ${indicators.get('ma_50', 0):.2f}")
print(f"  {'200-day MA':25s}: ${indicators.get('ma_200', 0):.2f}")
print(f"  {'Above 50-day MA':25s}: {indicators.get('above_50ma')}")
print(f"  {'Above 200-day MA':25s}: {indicators.get('above_200ma')}")
print(f"  {'Golden cross':25s}: {indicators.get('golden_cross')}")
print(f"  {'Death cross':25s}: {indicators.get('death_cross')}")

# ATR
print(f"\n  Volatility")
print(f"  {'ATR(14)':25s}: ${indicators.get('atr_14', 0):.2f}")

Technical Indicators for AAPL:

  Price
  Current price            : $272.14

  RSI
  RSI(14)                  : 56.9  [NEUTRAL]

  MACD
  MACD line                : 0.7646
  MACD signal              : 0.6538
  MACD histogram           : 0.1108
  Bullish crossover        : True
  Bearish crossover        : False

  Bollinger Bands
  Upper                    : $281.12
  Middle (20-day MA)       : $266.66
  Lower                    : $252.20
  %B (position in band)    : 0.69
  Below lower band         : False

  Moving Averages
  20-day MA                : $266.66
  50-day MA                : $265.57
  200-day MA               : $241.46
  Above 50-day MA          : True
  Above 200-day MA         : True
  Golden cross             : False
  Death cross              : False

  Volatility
  ATR(14)                  : $6.63


In [10]:
# Relative strength vs S&P 500
benchmark_df = fetcher.get_benchmark_data()
rs = calculate_relative_strength(df, benchmark_df)

rs_label = "OUTPERFORMING" if rs > 1.0 else "UNDERPERFORMING" if rs < 1.0 else "IN LINE"
print(f"Relative Strength vs S&P 500 (6-month): {rs:.3f}  [{rs_label}]")

Relative Strength vs S&P 500 (6-month): 1.122  [OUTPERFORMING]


---
## 4. Finnhub — Sentiment, Analysts, Price Targets, Insiders

**API key required:** `FINNHUB_API_KEY`  
**Rate limit:** 60 requests/min (free tier)

In [11]:
from src.agents.market_data.finnhub_client import FinnhubClient

finnhub = FinnhubClient()
print(f"Finnhub client initialised (base URL: {finnhub.base_url})")

Finnhub client initialised (base URL: https://finnhub.io/api/v1)


In [12]:
# 4a. News Sentiment
news_sentiment = finnhub.get_news_sentiment(TEST_TICKER)
print(f"News Sentiment for {TEST_TICKER}:")
pprint(news_sentiment)

[22:41:33] ERROR    Finnhub API error on /news-sentiment: Client error '403 Forbidden' for url                     
                    'https://finnhub.io/api/v1/news-sentiment?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&symbol
                    =AAPL'                                                                                         
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

[22:41:34] ERROR    Finnhub API error on /news-sentiment: Client error '403 Forbidden' for url                     
                    'https://finnhub.io/api/v1/news-sentiment?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&symbol
                    =AAPL'                                                                                         
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

[22:41:36] ERROR    Finnhub API error on /news-sentiment: Client error '403 Forbidden' for url                     
                    'https://finnhub.io/api/v1/news-sentiment?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&symbol
                    =AAPL'                                                                                         
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

[22:41:37] ERROR    Failed to get news sentiment for AAPL: RetryError[<Future at 0x11f342f50 state=finished raised 
                    HTTPStatusError>]

News Sentiment for AAPL:
{'error': 'RetryError[<Future at 0x11f342f50 state=finished raised '
          'HTTPStatusError>]',
 'symbol': 'AAPL'}


In [13]:
# 4b. Analyst Recommendations
analyst_recs = finnhub.get_analyst_recommendations(TEST_TICKER)
print(f"Analyst Recommendations for {TEST_TICKER}:")
pprint(analyst_recs)

Analyst Recommendations for AAPL:
{'buy': 21,
 'consensus': 'BUY',
 'hold': 17,
 'period': '2026-02-01',
 'sell': 2,
 'strong_buy': 14,
 'strong_sell': 0,
 'symbol': 'AAPL',
 'total_analysts': 54}


In [14]:
# 4c. Price Targets
price_targets = finnhub.get_price_targets(TEST_TICKER)
print(f"Price Targets for {TEST_TICKER}:")
pprint(price_targets)

[22:42:13] ERROR    Finnhub API error on /stock/price-target: Client error '403 Forbidden' for url                 
                    'https://finnhub.io/api/v1/stock/price-target?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&sy
                    mbol=AAPL'                                                                                     
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

[22:42:15] ERROR    Finnhub API error on /stock/price-target: Client error '403 Forbidden' for url                 
                    'https://finnhub.io/api/v1/stock/price-target?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&sy
                    mbol=AAPL'                                                                                     
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

[22:42:17] ERROR    Finnhub API error on /stock/price-target: Client error '403 Forbidden' for url                 
                    'https://finnhub.io/api/v1/stock/price-target?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&sy
                    mbol=AAPL'                                                                                     
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

           ERROR    Failed to get price targets for AAPL: RetryError[<Future at 0x11f4a7550 state=finished raised  
                    HTTPStatusError>]

Price Targets for AAPL:
{'error': 'RetryError[<Future at 0x11f4a7550 state=finished raised '
          'HTTPStatusError>]',
 'symbol': 'AAPL'}


In [15]:
# 4d. Insider Sentiment
insider = finnhub.get_insider_sentiment(TEST_TICKER)
print(f"Insider Sentiment for {TEST_TICKER}:")
pprint(insider)

Insider Sentiment for AAPL:
{'change': 0, 'month': 2, 'mspr': 0, 'symbol': 'AAPL', 'year': 2026}


In [16]:
# 4e. Company Peers
peers = finnhub.get_peers(TEST_TICKER)
print(f"Peers of {TEST_TICKER}: {peers}")

Peers of AAPL: ['AAPL', 'WDC', 'SNDK', 'DELL', 'HPE', 'PSTG', 'NTAP', 'SMCI', 'HPQ', 'IONQ', 'GPGI', 'DBD']


In [17]:
# 4f. Full Sentiment Bundle (combines all above)
full_sentiment = finnhub.get_full_sentiment_data(TEST_TICKER)
print(f"Full Finnhub Data for {TEST_TICKER}:")
print(json.dumps(full_sentiment, indent=2, default=str))

[22:42:41] ERROR    Finnhub API error on /news-sentiment: Client error '403 Forbidden' for url                     
                    'https://finnhub.io/api/v1/news-sentiment?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&symbol
                    =AAPL'                                                                                         
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

[22:42:42] ERROR    Finnhub API error on /news-sentiment: Client error '403 Forbidden' for url                     
                    'https://finnhub.io/api/v1/news-sentiment?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&symbol
                    =AAPL'                                                                                         
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

[22:42:44] ERROR    Finnhub API error on /news-sentiment: Client error '403 Forbidden' for url                     
                    'https://finnhub.io/api/v1/news-sentiment?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&symbol
                    =AAPL'                                                                                         
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

           ERROR    Failed to get news sentiment for AAPL: RetryError[<Future at 0x11f563e50 state=finished raised 
                    HTTPStatusError>]

[22:42:46] ERROR    Finnhub API error on /stock/price-target: Client error '403 Forbidden' for url                 
                    'https://finnhub.io/api/v1/stock/price-target?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&sy
                    mbol=AAPL'                                                                                     
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

[22:42:47] ERROR    Finnhub API error on /stock/price-target: Client error '403 Forbidden' for url                 
                    'https://finnhub.io/api/v1/stock/price-target?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&sy
                    mbol=AAPL'                                                                                     
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

[22:42:49] ERROR    Finnhub API error on /stock/price-target: Client error '403 Forbidden' for url                 
                    'https://finnhub.io/api/v1/stock/price-target?token=d6e7lghr01qh94m33u2gd6e7lghr01qh94m33u30&sy
                    mbol=AAPL'                                                                                     
                    For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

           ERROR    Failed to get price targets for AAPL: RetryError[<Future at 0x11f55eb90 state=finished raised  
                    HTTPStatusError>]

Full Finnhub Data for AAPL:
{
  "news_sentiment": {
    "error": "RetryError[<Future at 0x11f563e50 state=finished raised HTTPStatusError>]",
    "symbol": "AAPL"
  },
  "analyst_recommendations": {
    "symbol": "AAPL",
    "period": "2026-02-01",
    "strong_buy": 14,
    "buy": 21,
    "hold": 17,
    "sell": 2,
    "strong_sell": 0,
    "total_analysts": 54,
    "consensus": "BUY"
  },
  "price_targets": {
    "error": "RetryError[<Future at 0x11f55eb90 state=finished raised HTTPStatusError>]",
    "symbol": "AAPL"
  },
  "insider_sentiment": {
    "symbol": "AAPL",
    "mspr": 0,
    "change": 0,
    "month": 2,
    "year": 2026
  }
}


---
## 5. Alpha Vantage — AI-Powered News Sentiment

**API key required:** `ALPHA_VANTAGE_API_KEY`  
**Rate limit:** 25 requests/day (free tier) — use sparingly!

In [18]:
from src.agents.market_data.alpha_vantage_client import AlphaVantageClient

av = AlphaVantageClient()
print(f"Alpha Vantage client initialised (base URL: {av.base_url})")

Alpha Vantage client initialised (base URL: https://www.alphavantage.co/query)


In [19]:
# 5a. Ticker-specific news sentiment (costs 1 of your 25 daily requests)
ticker_news = av.get_market_news_sentiment(tickers=TEST_TICKER, limit=10)

if "error" not in ticker_news:
    print(f"Alpha Vantage Sentiment for {TEST_TICKER}:")
    print(f"  Total articles:    {ticker_news['total_articles']}")
    print(f"  Average sentiment: {ticker_news['average_sentiment']:.4f}")
    print(f"  Bullish articles:  {ticker_news['bullish_articles']}")
    print(f"  Bearish articles:  {ticker_news['bearish_articles']}")
    print(f"  Neutral articles:  {ticker_news['neutral_articles']}")
    print()
    print("Top 5 articles:")
    for i, art in enumerate(ticker_news.get('articles', [])[:5], 1):
        score = art['overall_sentiment_score']
        label = art['overall_sentiment_label']
        print(f"  {i}. [{label:12s} {score:+.3f}] {art['title'][:80]}")
        print(f"     Source: {art['source']}")
else:
    print(f"Error: {ticker_news}")

Alpha Vantage Sentiment for AAPL:
  Total articles:    10
  Average sentiment: 0.2824
  Bullish articles:  8
  Bearish articles:  0
  Neutral articles:  2

Top 5 articles:
  1. [Bullish      +0.437] Apple is bringing Mac Mini production to Houston
     Source: Axios
  2. [Neutral      +0.107] Citizens Financial Group Is Quietly Shifting Power To Its Customers
     Source: AD HOC NEWS
  3. [Bullish      +0.441] Apple to double Houston footprint, open AI training center for small businessess
     Source: Houston Chronicle
  4. [Bullish      +0.392] Garmin Ltd. Experiences Valuation Adjustment Amid Strong Market Performance and 
     Source: Markets Mojo
  5. [Somewhat-Bullish +0.338] Apple accelerates U.S. manufacturing, with Mac mini production coming later this
     Source: Apple


In [20]:
# 5b. Broad market sentiment (costs 1 of your 25 daily requests)
broad = av.get_broad_market_sentiment()

if "error" not in broad:
    print("Broad Market Sentiment (Alpha Vantage):")
    print(f"  Total articles:    {broad['total_articles']}")
    print(f"  Average sentiment: {broad['average_sentiment']:.4f}")
    print(f"  Bullish:           {broad['bullish_articles']}")
    print(f"  Bearish:           {broad['bearish_articles']}")
    print(f"  Neutral:           {broad['neutral_articles']}")
    
    # Show topic breakdown
    topic_counts = {}
    for art in broad.get('articles', []):
        for t in art.get('topics', []):
            topic_counts[t] = topic_counts.get(t, 0) + 1
    print(f"\n  Topic distribution:")
    for topic, count in sorted(topic_counts.items(), key=lambda x: -x[1])[:10]:
        print(f"    {topic:30s}: {count}")
else:
    print(f"Error: {broad}")

Broad Market Sentiment (Alpha Vantage):
  Total articles:    16
  Average sentiment: 0.1237
  Bullish:           8
  Bearish:           3
  Neutral:           5

  Topic distribution:
    economy_fiscal                : 16
    economy_monetary              : 16
    technology                    : 16
    earnings                      : 16
    financial_markets             : 14
    economy_macro                 : 2


---
## 6. Trading 212 — Account, Portfolio, Instruments

**API keys required:** `T212_API_KEY`, `T212_API_SECRET`  
**Mode:** Practice/Demo (`demo.trading212.com`)

In [21]:
from src.agents.execution.t212_client import T212Client

t212 = T212Client()
print(f"T212 client initialised (base URL: {t212.base_url})")

T212 client initialised (base URL: https://demo.trading212.com/api/v0)


In [22]:
# 6a. Account Cash Balance
try:
    cash = t212.get_cash()
    print("Account Cash:")
    pprint(cash)
except Exception as e:
    print(f"Error: {e}")

Account Cash:
{'blocked': 0,
 'free': 10000.0,
 'invested': 0,
 'pieCash': 0,
 'ppl': 0,
 'result': 0,
 'total': 10000.0}


In [23]:
# 6b. Account Info
try:
    info = t212.get_account_info()
    print("Account Info:")
    pprint(info)
except Exception as e:
    print(f"Error: {e}")

[22:43:41] WARNING  Rate limit low (0 remaining), pausing 2s

Account Info:
{'currencyCode': 'GBP', 'id': 32643382}


In [24]:
# 6c. Current Portfolio (open positions)
try:
    portfolio = t212.get_portfolio()
    if portfolio:
        print(f"Open Positions ({len(portfolio)}):")
        for pos in portfolio:
            ticker = pos.get('ticker', 'N/A')
            qty = pos.get('quantity', 0)
            avg_price = pos.get('averagePrice', 0)
            current = pos.get('currentPrice', 0)
            pnl = pos.get('ppl', 0)
            print(f"  {ticker:10s}  qty={qty:>8.2f}  avg={avg_price:>8.2f}  current={current:>8.2f}  P&L={pnl:>+8.2f}")
    else:
        print("No open positions (empty portfolio)")
except Exception as e:
    print(f"Error: {e}")

[22:44:01] WARNING  Rate limit low (0 remaining), pausing 2s

No open positions (empty portfolio)


In [25]:
# 6d. Available Instruments (first 20)
try:
    instruments = t212.get_instruments()
    print(f"Total instruments available: {len(instruments)}")
    print(f"\nFirst 20:")
    for inst in instruments[:20]:
        ticker = inst.get('ticker', 'N/A')
        name = inst.get('name', 'N/A')[:40]
        currency = inst.get('currencyCode', '?')
        print(f"  {ticker:15s} {currency:4s} {name}")
except Exception as e:
    print(f"Error: {e}")

[22:44:09] WARNING  Rate limit low (0 remaining), pausing 2s

Total instruments available: 16572

First 20:
  STN_US_EQ       USD  Stantec
  ZGYd_EQ         EUR  Fidelity National Information Services
  UFI_US_EQ       USD  Unifi
  C5MEd_EQ        EUR  Invesco S&P China A Midcap 500 Swap (Acc
  XNASl_EQ        USD  Xtrackers NASDAQ 100
  TLSd_EQ         EUR  Telia Co
  TMGl_EQ         GBX  Mission
  BORT_US_EQ      USD  Bank of Botetourt
  PETl_EQ         GBX  Petrel Resources
  5SIEl_EQ        GBX  Leverage Shares -5x Short 7-10 Year Trea
  JZ_US_EQ        USD  Jianzhi Education Technology
  SGD_US_EQ       USD  RenX Enterprises
  LGCY_US_EQ      USD  Legacy Education
  ONEG_US_EQ      USD  OneConstruction
  STRA_US_EQ      USD  Strategic Education
  FSZ_CA_EQ       CAD  Fiera Capital
  ETHPl_EQ        GBP  CoinShares Ethereum Staking
  HBT_US_EQ       USD  HBT Financial
  LCRd_EQ         EUR  Las Vegas Sands
  BRKR_US_EQ      USD  Bruker


In [26]:
# 6e. Pending Orders
try:
    pending = t212.get_pending_orders()
    print(f"Pending Orders ({len(pending)}):")
    for order in pending:
        pprint(order)
    if not pending:
        print("  (none)")
except Exception as e:
    print(f"Error: {e}")

[22:44:34] WARNING  Rate limit low (0 remaining), pausing 2s

Pending Orders (0):
  (none)


In [27]:
# 6f. Order History (last 10)
try:
    history = t212.get_order_history(limit=10)
    print("Recent Order History:")
    pprint(history)
except Exception as e:
    print(f"Error: {e}")

[22:44:38] WARNING  Rate limit low (0 remaining), pausing 2s

Recent Order History:
{'items': [], 'nextPagePath': None}


---
## 7. Sub-Strategies — Momentum, Mean Reversion, Factor

These are pure Python, no LLM calls. They score stocks based on technical indicators and fundamentals.

In [28]:
from src.agents.strategy.momentum import evaluate_momentum, MomentumSignal
from src.agents.strategy.mean_reversion import evaluate_mean_reversion, MeanReversionSignal
from src.agents.strategy.factor import calculate_factor_score, FactorScore, rank_by_factor

print("Strategy modules loaded")

Strategy modules loaded


In [29]:
# 7a. Momentum Strategy
mom_signal = evaluate_momentum(
    ticker=TEST_TICKER,
    indicators=indicators,
    relative_strength=rs,
    current_holding=False,  # Change to True to see sell signals for held positions
)

print(f"Momentum Signal for {TEST_TICKER}:")
print(f"  Action:    {mom_signal.action}")
print(f"  Score:     {mom_signal.score:.0f}/100")
print(f"  Reasoning: {mom_signal.reasoning}")

Momentum Signal for AAPL:
  Action:    BUY
  Score:     100/100
  Reasoning: Above 50-day MA | RSI in sweet spot (56.9) | MACD bullish crossover | RS vs S&P: 1.12


In [30]:
# 7b. Mean Reversion Strategy
mr_signal = evaluate_mean_reversion(
    ticker=TEST_TICKER,
    indicators=indicators,
    fundamentals=fundamentals,
    sector_avg_pe=25.0,  # Approximate sector average
    current_holding=False,
)

print(f"Mean Reversion Signal for {TEST_TICKER}:")
print(f"  Action:    {mr_signal.action}")
print(f"  Score:     {mr_signal.score:.0f}/100")
print(f"  Reasoning: {mr_signal.reasoning}")

Mean Reversion Signal for AAPL:
  Action:    HOLD
  Score:     3/100
  Reasoning: Positive earnings growth (18.3%) | High debt (102.6>1.5)


In [31]:
# 7c. Factor Strategy
# Calculate 6-month return for factor scoring
six_mo_return = None
if len(df) >= 126:
    six_mo_return = float(df['Close'].iloc[-1] / df['Close'].iloc[-126] - 1)

factor = calculate_factor_score(
    ticker=TEST_TICKER,
    fundamentals=fundamentals,
    indicators=indicators,
    relative_strength=rs,
    six_month_return=six_mo_return,
)

print(f"Factor Score for {TEST_TICKER}:")
print(f"  Composite: {factor.composite_score:.1f}/100")
print(f"  Value:     {factor.value_score:.1f}/100  (30% weight)")
print(f"  Quality:   {factor.quality_score:.1f}/100  (30% weight)")
print(f"  Momentum:  {factor.momentum_score:.1f}/100  (40% weight)")
print(f"  Reasoning: {factor.reasoning}")
print(f"  Components: {factor.components}")

Factor Score for AAPL:
  Composite: 71.0/100
  Value:     35.0/100  (30% weight)
  Quality:   75.0/100  (30% weight)
  Momentum:  95.0/100  (40% weight)
  Reasoning: High ROE (152.0%) | Strong margins (27.0%) | Strong 6mo return (20.0%) | RS vs S&P: 1.12
  Components: {'pe': 34.404552, 'pb': 45.37179, 'roe': 1.5202099, 'margin': 0.27037, 'debt_equity': 102.63, 'six_month_return': 0.2002937511390528, 'relative_strength': 1.121770249617459}


In [32]:
# 7d. Run all 3 strategies on multiple tickers to compare
COMPARE_TICKERS = ["AAPL", "MSFT", "GOOGL", "TSLA", "NVDA", "META", "AMZN", "JPM"]

print(f"{'Ticker':8s} | {'Mom Act':8s} {'Mom Score':10s} | {'MR Act':8s} {'MR Score':10s} | {'Factor':8s} {'V':5s} {'Q':5s} {'M':5s}")
print("-" * 95)

all_factors = []
for t in COMPARE_TICKERS:
    try:
        t_df = yf.download(t, period="1y", interval="1d", progress=False)
        if isinstance(t_df.columns, pd.MultiIndex):
            t_df.columns = t_df.columns.get_level_values(0)
        if t_df.empty or len(t_df) < 200:
            print(f"{t:8s} | Insufficient data")
            continue

        t_ind = calculate_indicators(t_df)
        t_fund = get_fundamentals(t)
        t_rs = calculate_relative_strength(t_df, benchmark_df)
        t_6mo = float(t_df['Close'].iloc[-1] / t_df['Close'].iloc[-126] - 1) if len(t_df) >= 126 else None

        t_mom = evaluate_momentum(t, t_ind, t_rs)
        t_mr = evaluate_mean_reversion(t, t_ind, t_fund)
        t_fac = calculate_factor_score(t, t_fund, t_ind, t_rs, t_6mo)
        all_factors.append(t_fac)

        print(f"{t:8s} | {t_mom.action:8s} {t_mom.score:8.0f}   | {t_mr.action:8s} {t_mr.score:8.0f}   | {t_fac.composite_score:6.1f}  {t_fac.value_score:4.1f} {t_fac.quality_score:4.1f} {t_fac.momentum_score:4.1f}")
    except Exception as e:
        print(f"{t:8s} | Error: {e}")

# Show factor ranking
if all_factors:
    print(f"\nFactor Ranking (top {len(all_factors)}):")
    for i, f in enumerate(rank_by_factor(all_factors, top_n=len(all_factors)), 1):
        print(f"  {i}. {f.ticker:8s} composite={f.composite_score:.1f}")

Ticker   | Mom Act  Mom Score  | MR Act   MR Score   | Factor   V     Q     M    
-----------------------------------------------------------------------------------------------


/Users/Kayvan/Library/Caches/pypoetry/virtualenvs/investment-agent-ZTjoZHpB-py3.11/lib/python3.11/site-packages/yfinance/scrapers/fundamentals.py:33: DeprecationWarning: 'Ticker.earnings' is deprecated as not available via API. Look for "Net Income" in Ticker.income_stmt.
  warnings.warn("'Ticker.earnings' is deprecated as not available via API. Look for \"Net Income\" in Ticker.income_stmt.", DeprecationWarning)


AAPL     | BUY           100   | HOLD            3   |   71.0  35.0 75.0 95.0


/Users/Kayvan/Library/Caches/pypoetry/virtualenvs/investment-agent-ZTjoZHpB-py3.11/lib/python3.11/site-packages/yfinance/scrapers/fundamentals.py:33: DeprecationWarning: 'Ticker.earnings' is deprecated as not available via API. Look for "Net Income" in Ticker.income_stmt.
  warnings.warn("'Ticker.earnings' is deprecated as not available via API. Look for \"Net Income\" in Ticker.income_stmt.", DeprecationWarning)


MSFT     | HOLD            0   | HOLD            8   |   50.5  60.0 75.0 25.0


...

/Users/Kayvan/Library/Caches/pypoetry/virtualenvs/investment-agent-ZTjoZHpB-py3.11/lib/python3.11/site-packages/yfinance/scrapers/fundamentals.py:33: DeprecationWarning: 'Ticker.earnings' is deprecated as not available via API. Look for "Net Income" in Ticker.income_stmt.
  warnings.warn("'Ticker.earnings' is deprecated as not available via API. Look for \"Net Income\" in Ticker.income_stmt.", DeprecationWarning)


META     | HOLD            0   | HOLD            3   |   47.5  50.0 75.0 25.0


/Users/Kayvan/Library/Caches/pypoetry/virtualenvs/investment-agent-ZTjoZHpB-py3.11/lib/python3.11/site-packages/yfinance/scrapers/fundamentals.py:33: DeprecationWarning: 'Ticker.earnings' is deprecated as not available via API. Look for "Net Income" in Ticker.income_stmt.
  warnings.warn("'Ticker.earnings' is deprecated as not available via API. Look for \"Net Income\" in Ticker.income_stmt.", DeprecationWarning)


AMZN     | HOLD            0   | HOLD            8   |   50.5  50.0 65.0 40.0
JPM      | HOLD            0   | HOLD           15   |   70.0  85.0 75.0 55.0

Factor Ranking (top 8):
  1. GOOGL    composite=75.5
  2. AAPL     composite=71.0
  3. JPM      composite=70.0
  4. NVDA     composite=53.0
  5. MSFT     composite=50.5
  6. AMZN     composite=50.5
  7. TSLA     composite=49.0
  8. META     composite=47.5


/Users/Kayvan/Library/Caches/pypoetry/virtualenvs/investment-agent-ZTjoZHpB-py3.11/lib/python3.11/site-packages/yfinance/scrapers/fundamentals.py:33: DeprecationWarning: 'Ticker.earnings' is deprecated as not available via API. Look for "Net Income" in Ticker.income_stmt.
  warnings.warn("'Ticker.earnings' is deprecated as not available via API. Look for \"Net Income\" in Ticker.income_stmt.", DeprecationWarning)


---
## 8. Strategy Engine — Claude Synthesis

This is the main LLM call. Claude takes all sub-strategy results + sentiment data and produces final trading decisions.

**API key required:** `ANTHROPIC_API_KEY`  
**Model:** `claude-sonnet-4-5-20250929`  
**Cost:** ~£0.01-0.03 per call

In [33]:
# 8a. First, see exactly what prompt Claude receives
from src.agents.strategy.prompts import STRATEGY_SYSTEM_PROMPT, build_strategy_prompt

print("=== SYSTEM PROMPT ===")
print(STRATEGY_SYSTEM_PROMPT)
print()
print("=" * 60)

=== SYSTEM PROMPT ===
You are an expert portfolio manager running an autonomous investment system.
Your goal is to outperform the S&P 500 by 10%+ over 6-12 months. You synthesize signals from
three quantitative strategies (Momentum, Mean Reversion, Factor) along with news sentiment data
to make final portfolio allocation decisions.

You must respond with ONLY valid JSON matching the exact schema specified. No markdown, no explanation outside the JSON.



In [34]:
# 8b. Build the exact user prompt that would be sent to Claude
# (using the data we already collected above)

sample_prompt = build_strategy_prompt(
    portfolio_state="Cash: £10,000 (100%). No open positions.",
    market_regime=f"Regime: {macro.get('market_regime', 'UNKNOWN')} | VIX: {macro.get('vix', 'N/A')}",
    momentum_proposals=f"- {TEST_TICKER}: {mom_signal.action} (score: {mom_signal.score:.0f}) — {mom_signal.reasoning}",
    mean_reversion_proposals=f"- {TEST_TICKER}: {mr_signal.action} (score: {mr_signal.score:.0f}) — {mr_signal.reasoning}",
    factor_proposals=f"- {TEST_TICKER}: composite={factor.composite_score:.0f} (V={factor.value_score:.0f} Q={factor.quality_score:.0f} M={factor.momentum_score:.0f}) — {factor.reasoning}",
    finnhub_sentiment=json.dumps(full_sentiment, indent=2, default=str)[:1000],
    alpha_vantage_sentiment=f"Avg sentiment: {ticker_news.get('average_sentiment', 'N/A')}" if 'error' not in ticker_news else "N/A",
    system_state="ACTIVE",
    vix=macro.get("vix"),
    cash_pct=100.0,
    max_position_pct=12.0,
    num_positions=0,
    max_positions=15,
    momentum_weight=0.35,
    mean_reversion_weight=0.30,
    factor_weight=0.35,
)

print("=== USER PROMPT (what Claude receives) ===")
print(sample_prompt[:3000])
if len(sample_prompt) > 3000:
    print(f"\n... [{len(sample_prompt)} total chars]")

=== USER PROMPT (what Claude receives) ===
Analyze the following portfolio state and strategy proposals. Make allocation decisions.

## CURRENT PORTFOLIO STATE
Cash: £10,000 (100%). No open positions.

## MARKET REGIME ASSESSMENT
Regime: BULL | VIX: 19.549999237060547

## STRATEGY PROPOSALS

### Momentum Strategy (weight: 0.35)
- AAPL: BUY (score: 100) — Above 50-day MA | RSI in sweet spot (56.9) | MACD bullish crossover | RS vs S&P: 1.12

### Mean Reversion Strategy (weight: 0.3)
- AAPL: HOLD (score: 3) — Positive earnings growth (18.3%) | High debt (102.6>1.5)

### Factor Strategy (weight: 0.35)
- AAPL: composite=71 (V=35 Q=75 M=95) — High ROE (152.0%) | Strong margins (27.0%) | Strong 6mo return (20.0%) | RS vs S&P: 1.12

## NEWS SENTIMENT DATA (Finnhub)
{
  "news_sentiment": {
    "error": "RetryError[<Future at 0x11f563e50 state=finished raised HTTPStatusError>]",
    "symbol": "AAPL"
  },
  "analyst_recommendations": {
    "symbol": "AAPL",
    "period": "2026-02-01",
    "strong

In [35]:
# 8c. Actually call Claude and see the response
# WARNING: This costs real money (~£0.01-0.03)
import anthropic

client = anthropic.Anthropic(api_key=settings.anthropic_api_key)

response = client.messages.create(
    model=settings.strategy_model,
    max_tokens=4096,
    system=STRATEGY_SYSTEM_PROMPT,
    messages=[{"role": "user", "content": sample_prompt}],
)

# Show usage
print(f"Model:         {settings.strategy_model}")
print(f"Input tokens:  {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")

from src.utils.cost_tracker import calculate_cost
cost = calculate_cost("anthropic", response.usage.input_tokens, response.usage.output_tokens)
print(f"Cost:          £{cost:.4f}")
print()

# Parse and display Claude's decision
raw = response.content[0].text
print("=== RAW RESPONSE ===")
print(raw[:2000])
print()

# Parse JSON
try:
    content = raw
    if "```json" in content:
        content = content.split("```json")[1].split("```")[0]
    elif "```" in content:
        content = content.split("```")[1].split("```")[0]
    decisions = json.loads(content)

    print("=== PARSED DECISIONS ===")
    print(f"Market Assessment: {decisions.get('market_assessment', 'N/A')}")
    print(f"Portfolio Commentary: {decisions.get('portfolio_commentary', 'N/A')}")
    print()
    for d in decisions.get('decisions', []):
        print(f"  {d['ticker']:8s} {d['action']:6s} alloc={d.get('target_allocation_pct', 0):.1f}%  "
              f"conviction={d.get('conviction', 0)}  strategy={d.get('primary_strategy', '?')}")
        print(f"           Reasoning: {d.get('reasoning', 'N/A')[:100]}")
        print()
except json.JSONDecodeError as e:
    print(f"Failed to parse JSON: {e}")

Model:         claude-sonnet-4-5-20250929
Input tokens:  1050
Output tokens: 800
Cost:          £0.0120

=== RAW RESPONSE ===
```json
{
  "market_assessment": "Market regime is BULL with VIX at 19.55, indicating moderate volatility within acceptable ranges for equity deployment. Broad market sentiment is positive (0.28 Alpha Vantage score), supporting risk-on positioning. Current conditions favor selective exposure to high-quality momentum names with strong fundamental backing.",
  "decisions": [
    {
      "ticker": "AAPL",
      "action": "BUY",
      "target_allocation_pct": 10.5,
      "conviction": 82,
      "primary_strategy": "momentum",
      "reasoning": "AAPL demonstrates exceptional multi-strategy alignment with perfect momentum score (100), strong factor composite (71), and bullish technical setup (RSI 56.9, MACD crossover, 1.12 relative strength vs S&P). Analyst consensus is solidly BUY with 35 buy/strong buy vs 2 sell ratings from 54 analysts, indicating strong professio

---
## 9. Moderation Panel — GPT-4o & Gemini Reviews

The Investment Committee: 3 LLMs review each trade proposal.

- **GPT-4o** (Skeptical Analyst): Challenges assumptions, flags risks  
- **Gemini 2.0 Flash** (Risk Assessor): Scores growth/risk/confidence  
- **Consensus**: 3 AGREE = APPROVED, 2 AGREE = CAUTION, 2 DISAGREE = BLOCKED

In [36]:
# Build a sample trade proposal (using Claude's decision if available, or a manual one)
sample_trade = {
    "ticker": TEST_TICKER,
    "action": "BUY",
    "target_allocation_pct": 5.0,
    "conviction": 75,
    "primary_strategy": "momentum",
    "reasoning": f"Strong momentum with RSI at {indicators.get('rsi_14', 0):.0f}, above 50-day MA, good relative strength vs S&P 500.",
    "growth_potential": "MEDIUM",
    "risk_level": "MEDIUM",
    "catalysts": ["Product launches", "AI integration"],
    "risks": ["Valuation stretch", "Macro headwinds"],
    "stop_loss_pct": -8.0,
    "upside_target_pct": 15.0,
}

# Try to use Claude's actual decision if we have one
try:
    if decisions and decisions.get('decisions'):
        sample_trade = decisions['decisions'][0]
        print(f"Using Claude's actual decision for {sample_trade['ticker']}")
except NameError:
    print(f"Using manual sample trade for {TEST_TICKER}")

portfolio_context = "Cash: £10,000 (100%). No open positions. Fresh portfolio."
sentiment_str = json.dumps(full_sentiment, indent=2, default=str)[:1500]

print(f"\nTrade proposal:")
pprint(sample_trade)

Using Claude's actual decision for AAPL

Trade proposal:
{'action': 'BUY',
 'catalysts': ['Continued momentum above 50-day MA with bullish MACD signals',
               'Strong analyst support (65% buy/strong buy ratings)',
               'Positive earnings growth trajectory (18.3%)',
               'Sustained relative strength vs market (1.12x)',
               'Potential product cycle catalysts in technology sector'],
 'conviction': 82,
 'exit_conditions': 'Exit if RSI exceeds 70 (overbought), breaks below 50-day '
                    'MA with volume, MACD bearish crossover confirmed, or '
                    'relative strength drops below 1.0 vs S&P',
 'expected_holding_period': '4-6 months',
 'growth_potential': 'HIGH',
 'news_sentiment_summary': 'Analyst consensus strongly bullish (BUY) with 65% '
                           'buy-side recommendations, though direct news '
                           'sentiment data unavailable; market-wide sentiment '
                           'pos

In [37]:
# 9a. GPT-4o Skeptical Review
# Cost: ~£0.005-0.01
from src.agents.moderation import openai_mod

print("=== GPT-4o System Prompt ===")
print(openai_mod.SYSTEM_PROMPT)
print()

gpt4o_result = openai_mod.review_trade(
    trade_proposal=sample_trade,
    portfolio_context=portfolio_context,
    sentiment_data=sentiment_str,
)

print("=== GPT-4o Verdict ===")
print(f"  Verdict:    {gpt4o_result.get('verdict')}")
print(f"  Reasoning:  {gpt4o_result.get('reasoning')}")
print(f"  Risk flags: {gpt4o_result.get('risk_flags', [])}")
print(f"  Mods:       {gpt4o_result.get('modifications')}")
print(f"  Available:  {gpt4o_result.get('available')}")

=== GPT-4o System Prompt ===
You are a skeptical investment analyst serving on an Investment Committee.
Your role is to challenge assumptions, identify risks the primary analyst may have missed,
and flag recency bias or overfitting to recent trends.

You have access to Finnhub sentiment data — use it to verify or challenge the thesis.

For each proposed trade, respond with ONLY valid JSON:
{
  "verdict": "AGREE|DISAGREE|MODIFY",
  "reasoning": "2-3 sentence specific reasoning",
  "risk_flags": ["list of specific risks identified"],
  "modifications": null or {"target_allocation_pct": X, "stop_loss_pct": Y}
}



[23:21:40] INFO     Cost logged: openai/gpt-4o - 909in/153out = £0.0030

=== GPT-4o Verdict ===
  Verdict:    MODIFY
  Reasoning:  While the momentum and analyst consensus are strong, the lack of news sentiment data and neutral insider sentiment reduce conviction. Elevated debt levels and market volatility are significant risks. A slightly lower allocation and tighter stop-loss could mitigate these risks.
  Risk flags: ['Elevated debt levels (102.6x ratio)', 'VIX at 19.55 indicates volatility risk', 'Neutral insider sentiment (MSPR 0)', 'Valuation compression risk if momentum falters']
  Mods:       {'target_allocation_pct': 8.0, 'stop_loss_pct': -5.0}
  Available:  True


In [38]:
# 9b. Gemini Risk Assessment
# Cost: ~£0.001
from src.agents.moderation import gemini_mod

print("=== Gemini System Prompt ===")
print(gemini_mod.SYSTEM_PROMPT)
print()

gemini_result = gemini_mod.review_trade(
    trade_proposal=sample_trade,
    portfolio_context=portfolio_context,
    sentiment_data=sentiment_str,
)

print("=== Gemini Verdict ===")
print(f"  Verdict:        {gemini_result.get('verdict')}")
print(f"  Growth score:   {gemini_result.get('growth_score')}/10")
print(f"  Risk score:     {gemini_result.get('risk_score')}/10")
print(f"  Confidence:     {gemini_result.get('confidence_score')}/10")
print(f"  Assessment:     {gemini_result.get('assessment')}")
print(f"  High risk flag: {gemini_result.get('high_risk_flag')}")
print(f"  Modifications:  {gemini_result.get('modifications')}")
print(f"  Available:      {gemini_result.get('available')}")

=== Gemini System Prompt ===
You are an independent risk assessor on an Investment Committee.
Score each proposed trade on three dimensions:
- Growth potential: 1-10
- Risk level: 1-10
- Confidence in thesis: 1-10

Flag any trade where risk > growth potential.
Consider news sentiment data in your scoring.

Respond with ONLY valid JSON:
{
  "verdict": "AGREE|DISAGREE|MODIFY",
  "growth_score": 7,
  "risk_score": 4,
  "confidence_score": 6,
  "assessment": "2-sentence independent assessment",
  "high_risk_flag": false,
  "modifications": null or {"target_allocation_pct": X}
}



[23:22:09] ERROR    Gemini moderation failed: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model        
                    models/gemini-2.0-flash is no longer available to new users. Please update your code to use a  
                    newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}

=== Gemini Verdict ===
  Verdict:        SKIP
  Growth score:   None/10
  Risk score:     None/10
  Confidence:     None/10
  Assessment:     None
  High risk flag: None
  Modifications:  None
  Available:      False


In [39]:
# 9c. Run the full Moderation Panel to see consensus logic
from src.agents.moderation.panel import ModerationPanel

panel = ModerationPanel()

mod_result = panel.review_trade(
    trade_proposal=sample_trade,
    portfolio_context=portfolio_context,
    sentiment_data=sentiment_str,
    conviction=sample_trade.get("conviction", 75),
    cycle_id="notebook_test",
)

print("=== MODERATION PANEL CONSENSUS ===")
print(f"  Ticker:        {mod_result.ticker}")
print(f"  CONSENSUS:     {mod_result.consensus}")
print(f"  Strategy:      {mod_result.strategy_verdict}")
print(f"  GPT-4o:        {mod_result.gpt4o_verdict.get('verdict') if mod_result.gpt4o_verdict else 'SKIP'}")
print(f"  Gemini:        {mod_result.gemini_verdict.get('verdict') if mod_result.gemini_verdict else 'SKIP'}")
print(f"  Moderators:    {mod_result.moderators_available}/2")
print(f"  Caution flag:  {mod_result.caution_flag}")

[23:37:41] INFO     Cost logged: openai/gpt-4o - 909in/150out = £0.0030

[23:37:42] ERROR    Gemini moderation failed: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model        
                    models/gemini-2.0-flash is no longer available to new users. Please update your code to use a  
                    newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}

=== MODERATION PANEL CONSENSUS ===
  Ticker:        AAPL
  CONSENSUS:     CAUTION
  Strategy:      AGREE
  GPT-4o:        MODIFY
  Gemini:        SKIP
  Moderators:    1/2
  Caution flag:  True


---
## 10. Risk Manager — Hard Rule Checks

Pure Python, no LLM calls. These rules have **VETO power** and can never be overridden by any LLM.

In [40]:
from src.agents.risk.risk_manager import RiskManager

risk_mgr = RiskManager()

# Print all risk limits from settings
print("Risk Manager Configuration:")
print(f"  Max single stock:    {settings.max_single_stock_pct}%")
print(f"  Max sector:          {settings.max_sector_pct}%")
print(f"  Max correlation:     {settings.max_correlation}")
print(f"  Cautious drawdown:   {settings.cautious_drawdown_pct}%")
print(f"  Halt drawdown:       {settings.halt_drawdown_pct}%")
print(f"  Daily loss halt:     {settings.daily_loss_halt_pct}%")
print(f"  Cash floor:          {settings.cash_floor_pct}%")
print(f"  VIX high:            {settings.vix_high}")
print(f"  VIX extreme:         {settings.vix_extreme}")
print(f"  Min positions:       {settings.min_positions}")

Risk Manager Configuration:
  Max single stock:    12.0%
  Max sector:          30.0%
  Max correlation:     0.7
  Cautious drawdown:   5.0%
  Halt drawdown:       15.0%
  Daily loss halt:     2.0%
  Cash floor:          10.0%
  VIX high:            25.0
  VIX extreme:         35.0
  Min positions:       5


In [41]:
# 10a. Test individual rules

# Max single stock
r1 = risk_mgr.check_max_single_stock("AAPL", 15.0, {"MSFT": 8.0})
print(f"Max single stock (15% AAPL): {r1.passed} — {r1.message}")
if r1.adjusted_allocation:
    print(f"  Adjusted to: {r1.adjusted_allocation}%")

r2 = risk_mgr.check_max_single_stock("AAPL", 8.0, {"MSFT": 8.0})
print(f"Max single stock (8% AAPL):  {r2.passed} — {r2.message}")
print()

Max single stock (15% AAPL): False — AAPL would be 15.0% of portfolio (max 12.0%)
  Adjusted to: 12.0%
Max single stock (8% AAPL):  True — AAPL at 8.0% within limit (12.0%)



In [42]:
# Max sector concentration
r3 = risk_mgr.check_max_sector("AAPL", "Technology", 10.0, {"Technology": 25.0})
print(f"Sector limit (Tech at 25% + 10%): {r3.passed} — {r3.message}")

r4 = risk_mgr.check_max_sector("AAPL", "Technology", 5.0, {"Technology": 20.0})
print(f"Sector limit (Tech at 20% + 5%):  {r4.passed} — {r4.message}")
print()

Sector limit (Tech at 25% + 10%): False — Sector 'Technology' would be 35.0% (max 30.0%). Only 5.0% room.
Sector limit (Tech at 20% + 5%):  True — Sector 'Technology' at 25.0% within limit (30.0%)



In [43]:
# VIX-based limits
r5 = risk_mgr.check_vix_limit(vix=15.0, proposed_pct=10.0)
print(f"VIX=15 (calm), 10% position:    {r5.passed} — {r5.message}")

r6 = risk_mgr.check_vix_limit(vix=28.0, proposed_pct=10.0)
print(f"VIX=28 (high), 10% position:    {r6.passed} — {r6.message}")

r7 = risk_mgr.check_vix_limit(vix=40.0, proposed_pct=10.0)
print(f"VIX=40 (extreme), 10% position: {r7.passed} — {r7.message}")
print()

VIX=15 (calm), 10% position:    True — VIX at 15.0, position size OK
VIX=28 (high), 10% position:    False — VIX high (28.0>25.0): max position 8.0%
VIX=40 (extreme), 10% position: False — VIX extreme (40.0>35.0): max position 5.0%



In [44]:
# Cash floor check
r8 = risk_mgr.check_cash_floor(cash_pct=50.0, trade_pct=10.0)
print(f"Cash 50%, trade 10%: {r8.passed} — {r8.message}")

r9 = risk_mgr.check_cash_floor(cash_pct=15.0, trade_pct=8.0)
print(f"Cash 15%, trade 8%:  {r9.passed} — {r9.message}")
print()

Cash 50%, trade 10%: True — Post-trade cash 40.0% above floor (10.0%)
Cash 15%, trade 8%:  False — Trade would leave cash at 7.0% (min 10.0%). Max trade: 5.0%



In [45]:
# Drawdown state check
r10 = risk_mgr.check_drawdown(current_value=9800, peak_value=10000)
print(f"2% drawdown:  {r10.passed} — {r10.message}")
print(f"  State: {risk_mgr.get_drawdown_state(9800, 10000)}")

r11 = risk_mgr.check_drawdown(current_value=9200, peak_value=10000)
print(f"8% drawdown:  {r11.passed} — {r11.message}")
print(f"  State: {risk_mgr.get_drawdown_state(9200, 10000)}")

r12 = risk_mgr.check_drawdown(current_value=8000, peak_value=10000)
print(f"20% drawdown: {r12.passed} — {r12.message}")
print(f"  State: {risk_mgr.get_drawdown_state(8000, 10000)}")
print()

2% drawdown:  True — Drawdown 2.0% within limits
  State: ACTIVE
8% drawdown:  True — CAUTIOUS: Drawdown 8.0% exceeds 5.0%
  State: CAUTIOUS
20% drawdown: False — HALT: Drawdown 20.0% exceeds 15.0%! Liquidate all.
  State: HALTED



In [46]:
# 10b. Full trade evaluation (all rules together)
verdict = risk_mgr.evaluate_trade(
    ticker=TEST_TICKER,
    action="BUY",
    proposed_allocation_pct=8.0,
    sector=fundamentals.get("sector", "Technology"),
    current_portfolio={"MSFT": 8.0, "GOOGL": 7.0, "NVDA": 6.0, "JPM": 5.0, "JNJ": 4.0},
    sector_allocations={"Technology": 21.0, "Financials": 5.0, "Healthcare": 4.0},
    portfolio_returns={},  # Empty — correlation check will be skipped
    current_value=9800,
    peak_value=10000,
    cash_pct=65.0,
    vix=macro.get("vix"),
    daily_pnl_pct=-0.5,
    daily_loss_halt_until=None,
    num_positions=5,
    system_state="ACTIVE",
    cycle_id="notebook_test",
)

print(f"=== RISK VERDICT for {TEST_TICKER} BUY 8% ===")
print(f"  Verdict:              {verdict.verdict}")
print(f"  Proposed allocation:  {verdict.proposed_allocation_pct:.1f}%")
print(f"  Adjusted allocation:  {verdict.adjusted_allocation_pct}%" if verdict.adjusted_allocation_pct else "  Adjusted allocation:  N/A")
print(f"  Rules checked:        {verdict.rules_checked}")
print(f"  Triggered rules:      {verdict.triggered_rules}")
print(f"  Reasoning:            {verdict.reasoning}")

=== RISK VERDICT for AAPL BUY 8% ===
  Verdict:              APPROVE
  Proposed allocation:  8.0%
  Adjusted allocation:  8.0%
  Rules checked:        ['max_single_stock', 'max_sector', 'vix_limit', 'cash_floor', 'daily_loss_halt', 'cautious_state', 'drawdown', 'correlation']
  Triggered rules:      []
  Reasoning:            All risk checks passed


---
## Summary

| Section | Data Source | API Key | Cost | Status |
|---------|-----------|---------|------|--------|
| 2 | yfinance | None | Free | Check cells above |
| 3 | Technical Indicators (ta) | None | Free | Check cells above |
| 4 | Finnhub | `FINNHUB_API_KEY` | Free tier (60 req/min) | Check cells above |
| 5 | Alpha Vantage | `ALPHA_VANTAGE_API_KEY` | Free tier (25 req/day) | Check cells above |
| 6 | Trading 212 | `T212_API_KEY` + `T212_API_SECRET` | Free (Demo) | Check cells above |
| 7 | Sub-strategies | None | Free (local) | Check cells above |
| 8 | Claude (Anthropic) | `ANTHROPIC_API_KEY` | ~£0.01-0.03/call | Check cells above |
| 9 | GPT-4o + Gemini | `OPENAI_API_KEY` + `GOOGLE_AI_API_KEY` | ~£0.005 + £0.001/call | Check cells above |
| 10 | Risk Manager | None | Free (local) | Check cells above |